# A LoKI batch

This notebook tells how a batch of reductions is run in the architecture sketch,
on the same LoKI@Larmor tutorial files as `loki-session.ipynb`. That notebook is
one person moving one parameter; this one is many samples reduced with the same
settings, first by hand and then by a rule as the runs arrive.

Four samples were measured, each with a transmission run just before it, against
one solvent background and one empty-beam run. Reducing all four to I(Q) is the
job an ISIS batch file does.

The vocabulary, in addition to that of `loki-session.ipynb`:

- A **template** is a stored, versioned partial request: every parameter except
  the ones that change per sample, which are its **blanks**.
- A **batch** is the records under one label, one **member** per sample. Nothing
  else is stored: the batch table is a query over the records.
- A **rule** is a template plus a **lookup** that fills blanks per dataset, a
  **selector** that says which datasets it applies to, and a **bound** below which
  it does not fire on its own.
- `apply` is the one operation that makes requests from a template or rule. The
  trigger loop, `backlog`, and `reprocess` all call it.

Tables are pandas frames throughout; the design document
`docs/developer/rules.md` has a section that maps each concept to its pandas
counterpart.

## The journal and the arriving runs

The tutorial files carry run numbers but no sample name or run role, which a rule
needs to tell a sample from a transmission run. At a facility the catalogue
declares these fields. Here a journal frame stands in for it and is handed to the
folder source, which attaches the fields to each dataset by run number.

The folder the source reads is a temporary one, and `arrive` links tutorial files
into it, so that the notebook can let runs arrive while a rule is active. The
ISIS polymer and its transmission run are held back for that.

In [ ]:
import tempfile
from pathlib import Path

import pandas as pd
import plopp as pp

from ess.apps import loki
from ess.apps.batch import (
    TriggerLoop,
    apply,
    backlog,
    batch_table,
    dataset_table,
    reprocess,
    shadowed,
    trigger_status,
)
from ess.apps.client import local
from ess.apps.rules import AsOf, Like, Lookup, LookupEntry, Rule, Selector, Template
from ess.apps.sources import FolderSource
from ess.apps.spec import dataset_ref
from ess.reduce.spec.parameters import QEdges

journal = pd.DataFrame.from_records(
    [
        (60384, 'porous silica', 'transmission'),
        (60385, 'porous silica', 'sample'),
        (60386, 'AgBeh', 'transmission'),
        (60387, 'AgBeh', 'sample'),
        (60388, 'deuterated SDS', 'transmission'),
        (60389, 'deuterated SDS', 'sample'),
        (60392, 'empty beam', 'empty-beam'),
        (60393, 'solvent', 'background'),
        (60394, 'ISIS polymer', 'transmission'),
        (60395, 'ISIS polymer', 'sample'),
    ],
    columns=['run', 'sample', 'role'],
).set_index('run')
journal

In [ ]:
cache = loki.cache()
root = Path(tempfile.mkdtemp(prefix='loki-batch-'))
incoming = root / 'incoming'
incoming.mkdir()


def arrive(*runs):
    """Let runs arrive: link their tutorial files into the folder the source reads."""
    for run in runs:
        (path,) = cache.glob(f'{run}-*.nxs')
        (incoming / path.name).symlink_to(path)


arrive(*journal.index.drop([60394, 60395]))

client = local(
    root / 'store',
    instrument='loki',
    proposal='p1',
    submitter='notebook',
    registry=loki.registry(),
    sources=[
        FolderSource(
            incoming,
            '*.nxs',
            identity=r'(?P<run>\d+)-.*',
            instrument='loki',
            journal=journal.to_dict('index'),
        )
    ],
)

dataset_table(client)

## The template

The beam centre is computed once, from the AgBeh run, as in `loki-session.ipynb`.
The template holds everything the four samples share, including a reference to
the beam-centre record. The two blanks are the sample run and its transmission
run. `dataset_field` names the blank a dataset fills when a rule applies the
template to one; it matters only in the second half.

In [ ]:
def run(number):
    return dataset_ref(instrument='loki', run=number)


center = client.run(loki.BEAM_CENTER, {'sample_run': run(60387)})

template = Template(
    name='loki-iofq-larmor',
    spec=loki.IOFQ.id,
    params={
        'background_run': run(60393),
        'background_transmission_run': run(60392),
        'empty_beam_run': run(60392),
        'direct_beam': dataset_ref(path=cache / 'direct-beam-loki-all-pixels.h5'),
        'beam_center': center.ref(),
        'q': QEdges(start=0.01, stop=0.3, num_bins=100),
    },
    blanks=('sample_run', 'sample_transmission_run'),
    dataset_field='sample_run',
)
template.id, template.blanks

## A batch by hand

The person fills the blanks in a table, one row per member, keyed by a name they
choose. This is the ISIS batch file. `apply` fills the template once per row and
returns the requests as a group; nothing exists in the record store yet.

In [ ]:
typed = pd.DataFrame(
    {
        'sample_run': [run(60385), run(60387), run(60389)],
        'sample_transmission_run': [run(60384), run(60386), run(60388)],
    },
    index=['porous silica', 'AgBeh', 'deuterated SDS'],
)
typed

Validation runs per request before anything is submitted: the schema, the
parameter values, and whether every referenced dataset and record can be found.
A form would show these errors next to the row.

In [ ]:
group = apply(client, template, typed=typed, label='samples')
pd.DataFrame(
    {key: client.validate(request).model_dump() for key, request in group.items()}
).T

In [ ]:
records = client.submit_group(group)
batch_table(client, 'samples')

The batch table is what was reduced with which values: one row per member, its
latest record, the template version that filled it, and the values of the fields
that differ per member. Those are the template's blanks and every field a member
typed; the `typed` column names the fields of the row a person typed. The table
is computed from the records each time it is asked for.

The outputs are ordinary record outputs, so plotting the batch is plotting a dict
of them.

In [ ]:
def curves(label, names=None):
    """The I(Q) of every member of a batch, keyed by member or by a name for it."""
    names = names or {}
    return {
        names.get(r.request.member_key, r.request.member_key): client.output(r, 'iofq')
        for r in client.batch(label)
    }


pp.plot(curves('samples'), norm='log')

## Correcting one member

A member is corrected by applying the template again for that member only, with
the corrected values typed. The new record carries the same label and member
key, so it supersedes the earlier one in the batch table. The earlier record is
still in the store.

In [ ]:
fix = pd.DataFrame(
    {
        'sample_run': [run(60389)],
        'sample_transmission_run': [run(60388)],
        'q': [QEdges(start=0.005, stop=0.3, num_bins=100)],
    },
    index=['deuterated SDS'],
)
client.submit_group(apply(client, template, typed=fix, label='samples'))
batch_table(client, 'samples')

In [ ]:
pd.DataFrame(
    [
        {'record': r.id, 'q_start': r.resolved_params['q']['start'], 'created': r.created}
        for r in client.records(label='samples', member_key='deuterated SDS')
    ]
)

## A rule

Typing the table by hand does not scale to a beamtime. A rule fills it from the
datasets as they arrive:

- The **selector** picks the datasets whose journal role is `sample`.
- The **lookup** fills the transmission run. Its one entry is an *as-of* fill:
  the nearest dataset before the member whose role is `transmission`. It is
  resolved against the member's own run number, not against the time of
  submission, so a late or repeated reduction gets the same transmission run.
- The template's `dataset_field` says that the selected dataset fills
  `sample_run`.

`Rule.over` sets the rule's bound to the newest dataset the source knows now, so
creating a rule mid-beamtime does not reduce everything measured so far by
surprise.

In [ ]:
rule = Rule.over(
    client.datasets(),
    name='iofq-auto',
    template=template,
    lookup=Lookup(
        name='transmission',
        entries=(
            LookupEntry(
                name='nearest-transmission',
                fills={
                    'sample_transmission_run': AsOf(
                        match={'role': Like(pattern='transmission')}
                    )
                },
            ),
        ),
    ),
    selector=Selector(match={'role': Like(pattern='sample')}),
)
rule.selector.after

`trigger_status` is the trigger loop's decision for one dataset, and why. It is
the same function the loop calls, so what a person reads is what the loop does.
Right now the loop fires on nothing: the samples lie at or before the bound.

In [ ]:
def trigger_table(rule):
    """The trigger status of every dataset the source knows."""
    return dataset_table(client).join(
        pd.DataFrame(
            [
                {'dataset': str(d.ref), **trigger_status(client, rule, d).model_dump()}
                for d in client.datasets()
            ]
        ).set_index('dataset')
    )


loop = TriggerLoop(client, rule)
trigger_table(rule)

### The backlog

The samples before the bound are offered as a backlog when the rule is created.
`backlog` returns a group like `apply` does, and nothing runs until the person
submits it. The rule's member keys are dataset identities, not names a person
chose. `dataset_table` is keyed by the same identities, so joining it onto the
batch table shows which sample each row is.

In [ ]:
def with_samples(table):
    """A rule's batch table with the sample name of each member."""
    samples = dataset_table(client)['sample']
    return table.join(samples).set_index('sample', append=True)


client.submit_group(backlog(client, rule))
with_samples(batch_table(client, rule))

A rule types nothing, so `typed` is empty in every row. The blanks show what the
rule filled: `sample_run` is the selected dataset, and `sample_transmission_run`
is what the as-of fill found, each sample's own transmission run.

### Runs arrive

The ISIS polymer is measured: first its transmission run, then the sample run.
The selector skips the transmission run; the sample run lies after the bound and
has no record under the rule's label, so the loop fires on it.

In [ ]:
arrive(60394, 60395)
trigger_table(rule).tail(2)

In [ ]:
fired = loop.run_once()
[(r.request.member_key, r.status.value) for r in fired]

The loop keeps no memory of what it fired on. Every clause of the trigger status
is a query over the records and the source, so a second pass, or a pass after a
restart, fires on nothing.

In [ ]:
loop.run_once()

In [ ]:
trigger_table(rule).tail(1)

In [ ]:
names = {str(run(n)): s for n, s in journal['sample'].items()}
pp.plot(curves('iofq-auto', names), norm='log')

### A correction, then a new rule version

A person corrects one member of the rule's batch the same way as in the batch by
hand: `apply` on the rule for that one dataset, with the corrected value typed.
Applying the rule, rather than the bare template, marks the request as the
rule's, which the rule's reserved label requires.

In [ ]:
polymer = next(d for d in client.datasets() if d.run == 60395)
client.submit_group(
    apply(
        client,
        rule,
        [polymer],
        {str(polymer.ref): {'q': QEdges(start=0.005, stop=0.3, num_bins=100)}},
    )
)
with_samples(batch_table(client, rule))

Later the instrument scientist changes the default Q binning. That is a new
template version and so a new rule version; the old versions stay, and each
record says which version made it.

`reprocess` offers the members whose latest record came from an older rule
version. It keeps what a person typed and fills everything else from the new
versions. That means the Q range typed for the polymer would silently win over
the new default. `shadowed` lists such cases, so that a person decides whether
the typed value still stands before submitting.

In [ ]:
rule_v2 = rule.revise(
    template=template.revise(q=QEdges(start=0.01, stop=0.3, num_bins=200))
)
shadowed(client, rule_v2, rule)

In [ ]:
client.submit_group(reprocess(client, rule_v2))
with_samples(batch_table(client, rule_v2))

The history of one member is an ordinary record query. The polymer has three
records under the rule's label: the loop's, the correction, and the reprocess.
The latest one is the batch table's row. Because nobody overrode the typed value
after reading `shadowed`, the reprocess kept the polymer's Q range and its 100
bins, while the other members moved to 200 bins.

In [ ]:
pd.DataFrame(
    [
        {
            'record': r.id,
            'rule': r.request.submission.rule,
            'typed': list(r.request.submission.typed),
            'q_start': r.resolved_params['q']['start'],
            'q_bins': r.resolved_params['q']['num_bins'],
        }
        for r in client.records(label='iofq-auto', member_key=str(polymer.ref))
    ]
)

In [ ]:
pp.plot(curves('iofq-auto', names), norm='log')